# Task 1 — Sentiment Analysis
**Internship Project by Venkata Sai Ajith**  
**May 2026**

## 1. Problem Statement
Develop a sentiment analysis tool that classifies text into **Positive / Negative / Neutral** for restaurant reviews.

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

## 2. Data Loading

In [ ]:
df = pd.read_csv('../data/sample_reviews.csv')
print(df.shape)
df.head()

## 3. Text Preprocessing (Lemmatization + NLTK)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

df['cleaned'] = df['Review'].apply(clean_text)
df.head()

## 4. Feature Extraction (TF-IDF)

In [ ]:
vectorizer = TfidfVectorizer(max_features=1500, ngram_range=(1,2))
X = vectorizer.fit_transform(df['cleaned']).toarray()
y = df['Liked']

## 5. Model Training & Comparison

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(C=5.0, max_iter=1000, random_state=42),
    'MultinomialNB': MultinomialNB(alpha=0.2),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42), cv=3)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results[name] = {'Accuracy': acc, 'F1-Score': f1}
    print(f"{name}: Accuracy = {acc:.4f}, F1 = {f1:.4f}")

best_model_name = max(results, key=lambda x: results[x]['F1-Score'])
print(f"\nBest Model: {best_model_name}")

## 6. Final Evaluation

In [ ]:
best_model = models[best_model_name]
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 7. Prediction Function

In [ ]:
def predict_sentiment(text):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = best_model.predict(vec)[0]
    return "Positive" if pred == 1 else "Negative"

print(predict_sentiment("The food was absolutely wonderful!"))
print(predict_sentiment("Terrible experience, never coming back."))

## 8. Model Deployment (Flask Web App)
The full Flask application is available in the `app/` folder. It provides:
- Web interface for real-time prediction
- REST API endpoint
- Example reviews for testing